# BreastMRISegmentator — run on a public demo case

This notebook walks through an end-to-end segmentation on a single DCE-MRI case using the per-phase NIfTI layout (**Layout B** in [`docs/usage.md`](../docs/usage.md)).

It is intentionally lightweight: it builds a tiny *synthetic* 4-phase case so the notebook runs anywhere (no real patient data, no download), then shows the exact commands you would use on a real case.

> **Note** — the actual nnU-Net inference cell requires the Exp4x weights (auto-downloaded from Zenodo on first use) and is best run on a GPU. It is marked clearly and can be skipped if you only want to see the input-preparation pipeline.

## 1. Install

In [ ]:
# From PyPI (once released):
# %pip install breast-mri-segmentator
#
# Or, from a local checkout of the repo:
# %pip install -e ..

import breast_mri_segmentator as bms
print('BreastMRISegmentator', bms.__version__)

## 2. Build a synthetic demo case

We fabricate four DCE phases for one case so the notebook is fully reproducible offline. On real data you would instead point `nifti_root` at your own `case/<case>_phase{N}.nii.gz` files (Layout B) or use `--from-dicom` (Layout A).

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import nibabel as nib

workdir = Path(tempfile.mkdtemp(prefix='bms_demo_'))
nifti_root = workdir / 'nifti_root'
case = nifti_root / 'demo_case'
case.mkdir(parents=True)

rng = np.random.default_rng(0)
shape = (32, 32, 16)
base = rng.normal(100, 10, size=shape).astype(np.float32)

# A crude enhancing 'lesion' that brightens after contrast then washes out.
lesion = np.zeros(shape, dtype=np.float32)
lesion[12:20, 12:20, 6:10] = 1.0

phase_gain = [0.0, 80.0, 60.0, 40.0, 30.0]  # P0..P4 enhancement of the lesion
for n, gain in enumerate(phase_gain):
    vol = base + gain * lesion
    nib.save(nib.Nifti1Image(vol, affine=np.eye(4)),
             str(case / f'demo_case_phase{n}.nii.gz'))

print('Wrote phases:', sorted(p.name for p in case.glob('*.nii.gz')))

## 3. Inspect the 4-channel kinetic input

`build_4ch_kinetic` turns the per-phase files into the nnU-Net channels `P0, P1, P1−P0, P_last−P1`. This is the same routine `segment()` calls internally; we run it here just to look at the channels.

In [ ]:
from breast_mri_segmentator.io import build_4ch_kinetic

kinetic_root = workdir / 'kinetic_4ch'
n = build_4ch_kinetic(nifti_root, kinetic_root)
print(f'Prepared {n} case(s)')

for ch in range(4):
    f = kinetic_root / f'demo_case_{ch:04d}.nii.gz'
    arr = nib.load(f).get_fdata()
    print(f'{f.name}: shape={arr.shape} mean={arr.mean():.2f} max={arr.max():.2f}')

## 4. Run segmentation (requires weights + GPU)

The cell below runs the real model. On first use it downloads the Exp4x checkpoint (~220 MB) from Zenodo into `~/.cache/breast_mri_segmentator/`. It needs `nnunetv2` installed and ideally a CUDA GPU.

On the synthetic case above the prediction is meaningless — the point is only to demonstrate the call. Set `RUN_INFERENCE = True` to execute it on your own real data.

In [ ]:
RUN_INFERENCE = False  # flip to True on a machine with weights + GPU

if RUN_INFERENCE:
    from breast_mri_segmentator import segment

    predictions = workdir / 'predictions'
    segment(
        input_dir=nifti_root,      # Layout B: per-phase NIfTI
        output_dir=predictions,
        from_dicom=False,
        use_tta=False,
    )
    out = predictions / 'demo_case.nii.gz'
    seg = nib.load(out).get_fdata()
    print('Labels present:', np.unique(seg))  # 0=bg 1=breast 2=FGT 3=tumor
else:
    print('Skipped inference. Equivalent CLI call:')
    print(f'  breast-mri-segment -i {nifti_root} -o {workdir / "predictions"}')

## 5. Cleanup

In [ ]:
import shutil
shutil.rmtree(workdir, ignore_errors=True)
print('Removed', workdir)